This notebook contains code used to train the v2 RF-DETR model on Kaggle (Epochs 0-19).

In [1]:
!nvidia-smi

Wed Jun  3 07:16:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%pip install -q rfdetr "rfdetr[loggers]" supervision torch faster-coco-eval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 112.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
from rfdetr import RFDETRMedium
from pathlib import Path
import torch

In [4]:
project_dir = Path.cwd().parent
Path('/kaggle/working/models/rfdetr_medium').mkdir(parents=True, exist_ok=True)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"Memory Cached: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

Using device: cuda
GPU Name: Tesla T4
Memory Allocated: 0.00 GB
Memory Cached: 0.00 GB


In [6]:
model = RFDETRMedium(num_classes=12)

model.train(
    dataset_dir=str(project_dir / 'input' / 'datasets' / 'b1leygr' 
                    / 'chessman-detection' / 'Chessman Detection.v5i.coco'),
    output_dir=str(project_dir / 'working' / 'models' / 'rfdetr_medium'),
    epochs=20,
    resolution=576,
    aug_config={
        "HorizontalFlip": {"p": 0.5}
    },
    checkpoint_interval=5,
    batch_size=4,
    grad_accum_steps=4,
    lr=1e-4,
    lr_scheduler='step',
    tensorboard=True,
    device=device
)

[2026-06-03 07:17:10] [INFO] rf-detr - Downloading pretrained weights for /root/.roboflow/models/rf-detr-medium.pth


/root/.roboflow/models/rf-detr-medium.pth:   0%|          | 0.00/386M [00:00<?, ?iB/s]

[2026-06-03 07:17:17] [INFO] rf-detr - MD5 validation successful for /root/.roboflow/models/rf-detr-medium.pth


[2026-06-03 07:17:17] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-06-03 07:17:17] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-06-03 07:17:18] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-medium.pth already exists with correct MD5 hash.


...

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Output()

[2026-06-03 07:17:45] [INFO] rf-detr - Best EMA mAP improved to 0.0086 (epoch 0)
[2026-06-03 07:47:06] [INFO] rf-detr - Best regular mAP saved to /kaggle/working/models/rfdetr_medium/checkpoint_best_regular.pth (epoch 0)
[2026-06-03 07:47:07] [INFO] rf-detr - Best EMA mAP improved to 0.5849 (epoch 0)
[2026-06-03 08:16:24] [INFO] rf-detr - Best regular mAP saved to /kaggle/working/models/rfdetr_medium/checkpoint_best_regular.pth (epoch 1)
[2026-06-03 08:16:25] [INFO] rf-detr - Best EMA mAP improved to 0.6315 (epoch 1)
[2026-06-03 08:46:31] [INFO] rf-detr - Best regular mAP saved to /kaggle/working/models/rfdetr_medium/checkpoint_best_regular.pth (epoch 2)
[2026-06-03 08:46:31] [INFO] rf-detr - Best EMA mAP improved to 0.6448 (epoch 2)
[2026-06-03 09:16:17] [INFO] rf-detr - Best regular mAP saved to /kaggle/working/models/rfdetr_medium/checkpoint_best_regular.pth (epoch 3)
[2026-06-03 09:16:17] [INFO] rf-detr - Best EMA mAP improved to 0.6539 (epoch 3)
[2026-06-03 09:45:15] [INFO] rf-det

`Trainer.fit` stopped: `max_epochs=20` reached.


[2026-06-03 17:11:42] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.6832, ema=0.6937)
